# Condition-Specific LoRA Face Editing

Apply trained skin-condition LoRA adapters to reference face images using **FLUX.2-klein-4B**. Configure paths and generation parameters in the first code cell, then run cells in order. The model and LoRA checkpoints are downloaded or supplied separately; this notebook does not train LoRAs.


In [ ]:
from pathlib import Path

# Paths (relative to the working directory, or replace with absolute paths)
BASE_IMAGES_DIR = Path("data/base_faces")
FACE_AREAS_FILE = Path("data/attributes/face_area.txt")
LORA_DIR = Path("models/condition_loras")
OUTPUT_DIR = Path("outputs/condition_faces")

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
BASE_SEED = 42
IMAGES_PER_CONDITION = 10
MAX_CONDITIONS = None  # None = all 12 conditions
MAX_LORAS_PER_CONDITION = 1  # None = all listed checkpoints per condition
WIDTH = HEIGHT = 1024
NUM_INFERENCE_STEPS = 6
GUIDANCE_SCALE = 1.0
MAX_SEQUENCE_LENGTH = 512
LORA_SCALE = 0.8
SCHEDULER_SHIFT = 3.0
SKIP_EXISTING = True
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


In [ ]:
# LoRA checkpoint filenames and training trigger words.
CONDITIONS = {
    'benign moles': [
        {'file': 'benignmolesface_flux2_klein_4b_lora.safetensors',
         'trigger': 'BenignMolesFACE'}
    ],
    'cafe-au-lait': [
        {'file': 'cafeaulaitface_flux2_klein_4b_lora.safetensors',
         'trigger': 'CafeAuLaitFACE'}
    ],
    'deep-acne scars': [
        {'file': 'deepacneface_flux2_klein_4b_lora.safetensors',
         'trigger': 'DeepacneFACE'}
    ],
    'freckles': [
        {'file': 'frecklesface_flux2_klein_4b_lora.safetensors',
         'trigger': 'FrecklesFace'}
    ],
    'healed scars': [
        {'file': 'healedscarface_flux2_klein_4b_lora.safetensors',
         'trigger': 'HealedscarFACE'}
    ],
    'keloid': [
        {'file': 'keloidface_flux2_klein_4b_lora.safetensors',
         'trigger': 'KeloidFace'}
    ],
    'melasma': [
        {'file': 'melasmaface_flux2_klein_4b_lora.safetensors',
         'trigger': 'MelasmaFACE'}
    ],
    'moles': [
        {'file': 'molesface_flux2_klein_4b_lora.safetensors',
         'trigger': 'MolesFACE'}
    ],
    'port-wine stains': [
        {'file': 'portwineface_flux2_klein_4b_lora.safetensors',
         'trigger': 'PortWineFACE'}
    ],
    'rosacea': [
        {'file': 'rosaceaface_flux2_klein_4b_lora.safetensors',
         'trigger': 'RosaceaFACE'}
    ],
    'tattoo': [
        {'file': 'tattooface_flux2_klein_4b_lora.safetensors',
         'trigger': 'FaceTattoo'}
    ],
    'vitiligo': [
        {'file': 'vitiligoface_flux2_klein_4b_lora.safetensors',
         'trigger': 'VitlitigoFace'}
    ]
}

In [ ]:
import re
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from diffusers import Flux2KleinPipeline, FlowMatchEulerDiscreteScheduler


def natural_sort_key(path):
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", path.name)]


def load_face_areas(path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing face-area file: {path}")
    areas = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if not line:
            continue
        line = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line)
        line = re.sub(r"\s*(?:[,;:=]\s*|\s+)(?:\d+(?:\.\d+)?%?|0?\.\d+)\s*$", "", line).strip()
        if line:
            areas.append(line)
    areas = list(dict.fromkeys(areas))
    if not areas:
        raise ValueError(f"No valid face areas found in {path}")
    return areas


def build_prompt(condition, trigger, face_area):
    return (
        f"{trigger}, Edit only the skin on the {face_area} by adding a realistic and "
        f"plausible, small size {condition} to the {face_area}. "
        "Blend it naturally with the surrounding skin and facial anatomy. "
        f"Preserve the person's identity, expression, facial structure, lighting, background, "
        f"and all areas outside the {face_area}."
    )


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", value).strip("_")


def load_adapter(pipe, checkpoint):
    pipe.load_lora_weights(str(checkpoint.parent), weight_name=checkpoint.name, adapter_name="skin")
    pipe.set_adapters("skin", adapter_weights=LORA_SCALE)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is required for this notebook.")
if WIDTH % 16 or HEIGHT % 16:
    raise ValueError("WIDTH and HEIGHT must be divisible by 16.")
if IMAGES_PER_CONDITION < 1 or (MAX_LORAS_PER_CONDITION is not None and MAX_LORAS_PER_CONDITION < 1):
    raise ValueError("Image and LoRA counts must be positive.")
if not BASE_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Missing base image directory: {BASE_IMAGES_DIR}")

base_images = sorted(
    (p for p in BASE_IMAGES_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
    key=natural_sort_key,
)
face_areas = load_face_areas(FACE_AREAS_FILE)
selected_conditions = list(CONDITIONS.items())[:MAX_CONDITIONS]
required = len(selected_conditions) * IMAGES_PER_CONDITION
if len(base_images) < required:
    raise ValueError(f"Expected at least {required} base images; found {len(base_images)}.")

for condition, checkpoints in selected_conditions:
    selected = checkpoints[:MAX_LORAS_PER_CONDITION]
    if not selected:
        raise ValueError(f"No LoRA checkpoints configured for {condition}.")
    for adapter in selected:
        path = LORA_DIR / adapter["file"]
        if not path.is_file():
            raise FileNotFoundError(f"Missing LoRA checkpoint for {condition}: {path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {len(selected_conditions)} conditions, {required} reference images.")


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(
    pipe.scheduler.config, shift=SCHEDULER_SHIFT
)
pipe.enable_model_cpu_offload()


In [ ]:
metadata_path = OUTPUT_DIR / "generation_metadata.csv"
columns = ["condition", "lora_checkpoint", "lora_trigger", "base_image", "face_area",
           "seed", "prompt", "output", "model", "lora_scale", "inference_steps",
           "guidance_scale", "scheduler_shift"]
existing = pd.read_csv(metadata_path).to_dict("records") if metadata_path.is_file() else []
rows_by_output = {row["output"]: row for row in existing}

for condition_idx, (condition, checkpoints) in enumerate(tqdm(selected_conditions, desc="Conditions")):
    start = condition_idx * IMAGES_PER_CONDITION
    images = base_images[start:start + IMAGES_PER_CONDITION]
    for lora_idx, adapter in enumerate(checkpoints[:MAX_LORAS_PER_CONDITION], start=1):
        checkpoint = LORA_DIR / adapter["file"]
        save_dir = OUTPUT_DIR / safe_name(condition) / f"lora_{lora_idx}"
        save_dir.mkdir(parents=True, exist_ok=True)
        # Do not load a LoRA when all requested outputs already exist.
        jobs = []
        for image_idx, base_path in enumerate(images):
            output_path = save_dir / f"{base_path.stem}.png"
            relative_output = output_path.relative_to(OUTPUT_DIR).as_posix()
            if SKIP_EXISTING and output_path.exists() and relative_output in rows_by_output:
                continue
            jobs.append((image_idx, base_path, output_path, relative_output))
        if not jobs:
            continue

        load_adapter(pipe, checkpoint)
        try:
            for image_idx, base_path, output_path, relative_output in tqdm(jobs, desc=condition, leave=False):
                face_area = face_areas[(image_idx + condition_idx) % len(face_areas)]
                # Keep seeds stable across runs and skipped files.
                index = (condition_idx * IMAGES_PER_CONDITION * len(checkpoints[:MAX_LORAS_PER_CONDITION])
                         + (lora_idx - 1) * IMAGES_PER_CONDITION + image_idx + 1)
                seed = BASE_SEED + index
                prompt = build_prompt(condition, adapter["trigger"], face_area)
                with Image.open(base_path) as source:
                    image = source.convert("RGB").resize((WIDTH, HEIGHT), Image.Resampling.LANCZOS)
                generator = torch.Generator(device="cuda").manual_seed(seed)
                result = pipe(
                    prompt=prompt, image=image, width=WIDTH, height=HEIGHT,
                    guidance_scale=GUIDANCE_SCALE, num_inference_steps=NUM_INFERENCE_STEPS,
                    generator=generator, max_sequence_length=MAX_SEQUENCE_LENGTH,
                )
                result.images[0].save(output_path)
                rows_by_output[relative_output] = {
                    "condition": condition,
                    "lora_checkpoint": checkpoint.name,
                    "lora_trigger": adapter["trigger"],
                    "base_image": base_path.name,
                    "face_area": face_area,
                    "seed": seed,
                    "prompt": prompt,
                    "output": relative_output,
                    "model": MODEL_ID,
                    "lora_scale": LORA_SCALE,
                    "inference_steps": NUM_INFERENCE_STEPS,
                    "guidance_scale": GUIDANCE_SCALE,
                    "scheduler_shift": SCHEDULER_SHIFT,
                }
                pd.DataFrame(list(rows_by_output.values()), columns=columns).to_csv(metadata_path, index=False)
        finally:
            pipe.unload_lora_weights()

print(f"Recorded {len(rows_by_output)} generated images in {metadata_path}.")
